# Step 6: 정규화 + 불용어 + 후처리 + 핵심/보조 분류
저장된 파일을 불러와서 CountVectorizer + c-TF-IDF를 1번만 재계산합니다.
- 불용어 (6-4 이상 토큰 추가 포함)
- 정규화 맵 (따듯→따뜻, xl→XL 등)
- CountVectorizer + c-TF-IDF 1회 재계산
- 후처리 (반복/범용/어순 중복 제거)
- 핵심/보조/기타 분류
## 입력
| 파일 | 내용 |
|------|------|
| `step3_1_embeddings.npy` ~ `step5_2_hdbscan_model.pkl` | 1_step3_5.ipynb 출력 |
## 출력 (01_부터 시작)
| 파일 | 내용 |
|------|------|
| `step6_1_topic_keywords.csv` | 카테고리 포함 최종 키워드 |
| `step6_4_topic_id_order.npy` | 토픽 순서 |
| `step6_5_count_matrix.npz` | Count 행렬 |
| `step6_6_words.npy` | 단어 목록 |
| `step6_7_vectorizer.pkl` | CountVectorizer |
| `step6_8_ctfidf.pkl` | c-TF-IDF 행렬 + 단어 + 토픽 매핑 |
| `step6_9_stopwords.pkl` | 불용어 셋 |
| `step6_10_normalize_map.pkl` | 정규화 맵 |
| `step6_11_core_attributes.pkl` | 핵심 속성어 |
| `step6_12_secondary_terms.pkl` | 보조 표현 |
| `step6_13_generic_attrs.pkl` | 범용 속성어 |


In [12]:
# 설치
!pip install kiwipiepy bertopic scikit-learn pandas numpy scipy

In [13]:
from google.colab import drive

import os
import time
import json
import pickle
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, save_npz
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.vectorizers import ClassTfidfTransformer
from kiwipiepy import Kiwi

In [14]:
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/sparta/tp4/review_analysis/data/'
INPUT_PATH = OUTPUT_DIR + 'embedding_input.parquet'

print(f'INPUT     : {INPUT_PATH}')
print(f'OUTPUT_DIR: {OUTPUT_DIR}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
INPUT     : /content/drive/MyDrive/sparta/tp4/review_analysis/data/embedding_input.parquet
OUTPUT_DIR: /content/drive/MyDrive/sparta/tp4/review_analysis/data/


In [15]:
# 저장된 파일 로드
# Step 3~5의 저장된 파일 불러오기
df = pd.read_parquet(INPUT_PATH)
df['리뷰내용_clean'] = df['리뷰내용_clean'].fillna('').astype(str)
df = df[df['리뷰내용_clean'].str.len() > 0].reset_index(drop=True)
docs = df['리뷰내용_clean'].tolist()
cluster_labels = np.load(OUTPUT_DIR + 'step5_1_cluster_labels.npy')
print(f'docs: {len(docs):,}건')
print(f'cluster_labels: {cluster_labels.shape}')
print(f'토픽 수 (outlier 제외): {len(set(cluster_labels)) - 1}개')

docs: 598,301건
cluster_labels: (598301,)
토픽 수 (outlier 제외): 68개


## 6-1. 보호어(사용자 사전) 정의
패션 핵심어를 카테고리별로 분류 등록.

In [16]:
PROTECTED_WORDS = {
    # 1. 핏/실루엣
    'fit': [
        '오버핏', '오버사이즈', '루즈핏', '레귤러핏', '슬림핏', '와이드핏',
        '세미오버', '박시핏', '머슬핏', '크롭핏', '박시', '핏',
        '벌룬핏', '부츠컷', '스탠다드핏', '세미와이드', '어깨선',
    ],
    # 2. 디테일/구조
    'detail': [
        '시보리', '워싱', '레이어드', '맨투맨', '후드티', '후드집업', '바람막이', '카라', '넥라인',
        '봉제', '스티치',
        '주머니', '포켓', '밑단', '소매', '배색', '단추', '지퍼', '자수', '프린팅', '밴딩',
        '목폴라', '반폴라', '투웨이', '스트링',
    ],
    # 3. 원단/품질
    'fabric': [
        '재질', '두께', '두께감', '기장', '마감', '퀄리티', '원단',
        '신축성', '내구성', '보풀', '촉감', '터치감',
        '기모', '린넨', '코듀로이', '광택', '탄탄', '흐물', '비침', '까슬',
    ],
    # 4. 색상
    'color': [
        '색감', '톤다운', '톤업', '채도', '색상',
        '실물', '화면', '사진', '원색', '파스텔', '형광', '무채색', '쨍함',
    ],
    # 5. 가격/가치
    'value': [
        '가성비', '가심비',
        '할인', '세일', '정가', '저렴', '비쌈', '돈값', '혜자', '득템', '창렬',
    ],
    # 6. 사이즈
    'size': [
        '핏감', '사이즈감',
        '정사이즈', '반사이즈', '작게', '크게', '허리', '총장', '팔길이', '끼임', '넉넉함', '밑위',
        's', 'm', 'l', 'xl', '2xl', '3xl'
    ],
}
all_protected = [w for words in PROTECTED_WORDS.values() for w in words]
PROTECTED_SET = set(all_protected)
print(f'보호어 총 {len(all_protected)}개')
print(f'카테고리: {list(PROTECTED_WORDS.keys())}')

with open(OUTPUT_DIR + 'protected_words.json', 'w', encoding='utf-8') as f:
    json.dump(PROTECTED_WORDS, f, ensure_ascii=False, indent=2)
print(f'저장 완료: {OUTPUT_DIR}protected_words.json')

보호어 총 104개
카테고리: ['fit', 'detail', 'fabric', 'color', 'value', 'size']
저장 완료: /content/drive/MyDrive/sparta/tp4/review_analysis/data/protected_words.json


## 6-2. kiwipiepy 토크나이저 구성

In [17]:
kiwi = Kiwi()
# 모든 보호어를 일반명사(NNG)로 사용자 사전 등록
# score=5: 우선순위 가중치 (kiwi 기본 0, 양수일수록 우선)
for word in all_protected:
    kiwi.add_user_word(word, 'NNG', score=5)
print(f'사용자 사전에 {len(all_protected)}개 단어 등록 완료')

사용자 사전에 104개 단어 등록 완료


## 6-3. 불용어 정의

In [18]:
# 6-4 (이상 토큰) + 6-5 (빈도 검증) 통합
STOPWORDS = {
    # 일반어
    '생각', '정도', '부분', '원하', '나오', '보이',
    '구매', '추천', '제품', '느낌',
    # 일반 감성어 (패션 속성어인 따뜻/편하/무난/어울리는 보존)
    '이쁘', '예쁘', '만족', '괜찮', '마음',
    # 데일리 동사
    '다니',
    # 이상 토큰 (6-4에서 확인)
    '모이',    # 기모이 오분절
    '목포',    # 팔목포인트 오분절
    '시키',    # 주문시키다 활용형
    '지사',    # 라지사이즈 오분절
    '댕기',    # 입고댕기다 구어체
    '구장',    # 주구장창 오분절
    '사고',    # 사다 명사형
    '드리',    # 추천드리다 어미 잔재
    # 빈도 검증 추가 (6-5)
    '자연',    # 자연스럽다 어근
    '주문',    # 구매와 동의어
    '구입',    # 구매와 동의어
    '이번',    # 일반 부사
    '요즘',    # 일반 부사
    '상품',    # 모든 리뷰의 자명한 대상
    # 추가
    '후기', '리뷰',
    # 브랜드
    '제멋', 'JEMUT',
    '필루미네이트','FILLUMINATE',
    '트래블','TRAVEL',
    # 추가2
    '감사','입' # 무신사 패션 리뷰에서 입(mouth)이 독립 토큰으로 의미 있게 등장할 맥락이 없기에 입다의 입을 불용어로 처리
}

# 보호어와 충돌 체크
conflict = STOPWORDS & PROTECTED_SET
assert not conflict, f'보호어와 불용어 충돌: {conflict}'
print(f'불용어: {len(STOPWORDS)}개')
print(f'보호어와 충돌: 없음 ✓')

불용어: 40개
보호어와 충돌: 없음 ✓


## 6-4. 정규화 맵 정의

In [19]:
NORMALIZE_MAP = {
    # 따뜻 변형 통합
    '따듯':   '따뜻',
    '따듯하': '따뜻',
    '따뜻하': '따뜻',
    # 편하 변형 통합
    '편안':   '편하',
    '편함':   '편하',
    '편안하': '편하',
    # 무난 변형 통합
    '무난하': '무난',
    '무난함': '무난',
    # 어울리 변형 통합
    '어울려': '어울리',
    '어울림': '어울리',
    '어울린': '어울리',
    # 분절 오류 교정
    '기모이': '기모',
    '목포':   '',
    # 사이즈 표기 통합 (M)
    '미듐':   '미디움',
    '미디엄': '미디움',
    # 사이즈 표기 통합 (L/XL)
    '엑라':   '엑스라지',
    '엑스':   '엑스라지',
    '라지로': '라지',
    # 시원 변형 통합
    '시원하': '시원',
    # 오타 교정
    '차코':   '차콜',
}
print(f'정규화 매핑: {len(NORMALIZE_MAP)}개')
for k, v in NORMALIZE_MAP.items():
    arrow = '→ (제거)' if v == '' else f'→ {v}'
    print(f'  {k:8} {arrow}')

정규화 매핑: 20개
  따듯       → 따뜻
  따듯하      → 따뜻
  따뜻하      → 따뜻
  편안       → 편하
  편함       → 편하
  편안하      → 편하
  무난하      → 무난
  무난함      → 무난
  어울려      → 어울리
  어울림      → 어울리
  어울린      → 어울리
  기모이      → 기모
  목포       → (제거)
  미듐       → 미디움
  미디엄      → 미디움
  엑라       → 엑스라지
  엑스       → 엑스라지
  라지로      → 라지
  시원하      → 시원
  차코       → 차콜


## 6-5. tokenize_ko 정의

In [20]:
ALLOWED_POS = {'NNG', 'NNP', 'VV', 'VA', 'XR', 'SL'}
# NNG: 일반명사, NNP: 고유명사, VV: 동사, VA: 형용사, XR: 어근, SL: 외국어

# tokenize_ko: 정규화 + 보호어 + 불용어
def tokenize_ko(text: str) -> list:
    """보호어 사전 + 영문 소문자 통일 + 동의어 정규화 + 불용어 적용"""
    if not text or not isinstance(text, str):
        return []
    tokens = []
    for token in kiwi.tokenize(text, normalize_coda=True):
        if token.tag not in ALLOWED_POS:
            continue
        word = token.form
        # 1) 영문 소문자 통일 (XL→xl, Ma→ma, MA→ma 등)
        if token.tag == 'SL':
            word = word.lower()
        # 2) 동의어 정규화
        word = NORMALIZE_MAP.get(word, word)
        # 정규화 결과 빈 문자열이면 제외
        if not word:
            continue
        # 3) 길이 1 제외 (보호어는 통과)
        if len(word) <= 1 and word not in PROTECTED_SET:
            continue
        # 4) 불용어 제외
        if word in STOPWORDS:
            continue
        tokens.append(word)
    return tokens
# 검증
test_sentences = [
    '팔목포인트가 좋아요',
    '기모이지만 따뜻하지 않아요',
    '따듯하고 편한데 무난하고 잘 어울려요',
    '미듐 시켰는데 라지로 살걸',
    '엑라가 너무 크네요 엑스로 갈걸',
    '이번 여름에 주문한 상품 만족',
    'XL 사이즈 구매했어요',   # XL → xl
    'Ma 브랜드 처음 입어봤어요',  # Ma → ma
    'MA 오버핏 좋아요',  # MA → ma
]
print('[tokenize_ko 검증]\n')
for s in test_sentences:
    result = tokenize_ko(s)
    print(f'원문: {s}')
    print(f'  토큰: {result}\n')

[tokenize_ko 검증]

원문: 팔목포인트가 좋아요
  토큰: ['팔목', '포인트']

원문: 기모이지만 따뜻하지 않아요
  토큰: ['기모', '따뜻']

원문: 따듯하고 편한데 무난하고 잘 어울려요
  토큰: ['따뜻', '편하', '무난', '어울리']

원문: 미듐 시켰는데 라지로 살걸
  토큰: ['미디움', '라지']

원문: 엑라가 너무 크네요 엑스로 갈걸
  토큰: ['엑스라지', '엑스라지']

원문: 이번 여름에 주문한 상품 만족
  토큰: ['여름']

원문: XL 사이즈 구매했어요
  토큰: ['xl', '사이즈']

원문: Ma 브랜드 처음 입어봤어요
  토큰: ['ma', '브랜드', '처음']

원문: MA 오버핏 좋아요
  토큰: ['ma', '오버핏']



## 6-6. CountVectorizer + c-TF-IDF 재계산

In [21]:
# outlier 제외 topic_id_order 생성
cluster_sizes = pd.Series(cluster_labels[cluster_labels != -1]).value_counts()
topic_id_order = sorted(set(cluster_labels.tolist()) - {-1})
print(f'토픽 수: {len(topic_id_order)}개 (outlier 제외)')
# 토큰화 1회만 실행 (ngram 포함)
print('\n토큰화 중...')
t0 = time.time()
tokenized_docs = [tokenize_ko(doc) for doc in docs]
print(f'토큰화 완료: {(time.time()-t0)/60:.1f}분')
# unigram + bigram 생성
def make_ngrams(tokens, n=2):
    unigrams = tokens
    bigrams = [' '.join(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    return unigrams + bigrams
print('ngram 생성 중...')
tokenized_docs_ngram = [make_ngrams(tokens) for tokens in tokenized_docs]
print(f'ngram 생성 완료')
def identity(x):
    return x
# min_df=50으로 최종 계산
FINAL_MIN_DF = 20
print(f'\nCountVectorizer + c-TF-IDF 계산 중... (min_df={FINAL_MIN_DF})')
t0 = time.time()
vectorizer = CountVectorizer(
    analyzer=identity,
    min_df=FINAL_MIN_DF,
    max_df=0.9,
)
doc_term_matrix = vectorizer.fit_transform(tokenized_docs_ngram)
words = vectorizer.get_feature_names_out()
print(f'  단어 수: {len(words):,}개')
# 토픽별 합산 (outlier 제외)
topic_term_rows = []
for tid in topic_id_order:
    mask = (cluster_labels == tid)
    summed = doc_term_matrix[mask].sum(axis=0)
    topic_term_rows.append(np.asarray(summed).flatten())
count_matrix = csr_matrix(np.vstack(topic_term_rows))
print(f'  행렬: {count_matrix.shape}  ← outlier 제외')
ctfidf_model = ClassTfidfTransformer(reduce_frequent_words=True)
ctfidf_matrix = ctfidf_model.fit_transform(count_matrix)
ctfidf_array = ctfidf_matrix.toarray()
TOP_N_RAW = 30
topic_keywords_raw = {}
for i, topic_id in enumerate(topic_id_order):
    scores = ctfidf_array[i]
    top_indices = scores.argsort()[::-1][:TOP_N_RAW]
    keywords = [(words[idx], scores[idx]) for idx in top_indices if scores[idx] > 0]
    topic_keywords_raw[topic_id] = keywords
print(f'\n완료: {(time.time()-t0)/60:.1f}분')
print(f'토픽 수: {len(topic_keywords_raw)}개')

토픽 수: 68개 (outlier 제외)

토큰화 중...
토큰화 완료: 13.1분
ngram 생성 중...
ngram 생성 완료

CountVectorizer + c-TF-IDF 계산 중... (min_df=20)
  단어 수: 14,651개
  행렬: (68, 14651)  ← outlier 제외

완료: 0.1분
토픽 수: 68개


/usr/local/lib/python3.12/dist-packages/bertopic/vectorizers/_ctfidf.py:82: RuntimeWarning: divide by zero encountered in divide
  idf = np.log((avg_nr_samples / df) + 1)


## 6-7. 후처리 필터 정의

In [22]:
# 후처리 필터 함수
def is_repeat_bigram(word):
    """'바지 바지', '사이즈 사이즈' 같은 반복 bigram 감지"""
    parts = word.split(' ')
    return len(parts) == 2 and parts[0] == parts[1]
GENERIC_ATTRS = {
    '편하', '편안',
    '따뜻', '따듯',
    '무난', '어울리',
    '적당', '괜찮',
    '좋',
}
def is_generic_pair(word):
    """범용 속성어끼리의 결합 bigram 감지"""
    parts = word.split(' ')
    if len(parts) != 2:
        return False
    return parts[0] in GENERIC_ATTRS and parts[1] in GENERIC_ATTRS
def bigram_key(word):
    """'편하 사이즈'와 '사이즈 편하'를 같은 키로 (어순 중복 제거용)"""
    parts = word.split(' ')
    if len(parts) == 2:
        return ' '.join(sorted(parts))
    return word
# 검증
test_cases = [
    ('바지 바지',   is_repeat_bigram, True),
    ('편하 따뜻',   is_generic_pair,  True),
    ('따뜻 편하',   is_generic_pair,  True),
    ('두툼 따뜻',   is_generic_pair,  False),
    ('사이즈 편하', is_generic_pair,  False),
]
print('[필터 함수 검증]')
for word, fn, expected in test_cases:
    result = fn(word)
    ok = '✓' if result == expected else '✗'
    print(f'  {ok} {fn.__name__}({word!r:18}) = {result}')

[필터 함수 검증]
  ✓ is_repeat_bigram('바지 바지'           ) = True
  ✓ is_generic_pair('편하 따뜻'           ) = True
  ✓ is_generic_pair('따뜻 편하'           ) = True
  ✓ is_generic_pair('두툼 따뜻'           ) = False
  ✓ is_generic_pair('사이즈 편하'          ) = False


## 6-8. 후처리 적용

In [23]:
# 후처리 적용
TOP_N_FINAL = 15
topic_keywords = {}
for topic_id in topic_id_order:
    raw = topic_keywords_raw[topic_id]
    filtered = []
    seen_keys = set()
    for word, score in raw:
        if score <= 0:
            continue
        # 필터 1: 반복 bigram 제거
        if is_repeat_bigram(word):
            continue
        # 필터 2: 범용 속성어끼리 결합 제거
        if is_generic_pair(word):
            continue
        # 필터 3: 어순 중복 제거
        key = bigram_key(word)
        if key in seen_keys:
            continue
        seen_keys.add(key)
        filtered.append((word, score))
        if len(filtered) >= TOP_N_FINAL:
            break
    topic_keywords[topic_id] = filtered
print(f'후처리 완료: {len(topic_keywords)}개 토픽')

후처리 완료: 68개 토픽


## 6-9. 핵심 / 보조 / 기타 분류

In [24]:
# 핵심 속성어
CORE_ATTRIBUTES = set(all_protected) | {
    # 색상
    '색깔', '컬러', '선명', '진하', '연하', '흐리', '탁하',
    '화사', '산뜻', '오묘', '은은', '피그먼트',
    # 신체 부위
    '허리', '허벅지', '엉덩이', '어깨', '소매', '밑단', '어깨선',
    '팔목', '손목', '몸통', '머리',
    # 사이즈 표기
    '치수', '사이즈', '길이', '품',
    # 원단/감촉
    '두툼', '도톰', '보온', '포근', '쫀쫀', '탄탄', '튼튼',
    '두께', '소재', '원단', '촉감',
    # 시즌
    '여름', '겨울', '봄', '가을', '간절기', '환절기', '쌀쌀',
    '시원', '날씨', '봄가을', '초겨울', '봄철', '초봄',
    # 디테일
    '벨트', '스트링', '밴딩', '주머니', '지퍼', '로고',
    '프린팅', '패딩', '기모', '카라', '체크', '빈티지',
    # 가격
    '가격', '저렴', '할인', '세일', '쿠폰', '대비',
    # 품목
    '바지', '후드티', '후드', '셔츠', '청바지', '데님', '반팔',
    '맨투맨', '티셔츠', '반바지', '바람막이', '카고', '벌룬',
    '팬츠', '아우터', '이너', '단독',
    # 배송
    '배송', '빠르', '느리', '걸리', '도착', '택배',
    # 세탁/관리
    '세탁', '빨래', '변형', '먼지', '냄새', '실밥', '늘어나', '줄어들', '건조기',
    # 사이즈 평가
    '타이트', '답답', '쫀쫀',
    # 디자인
    '디자인', '디테일',
    # 부정 평가
    '아쉬움', '실망', '후회', '불편',
}
# 보조 표현
SECONDARY_TERMS = {
    '편하', '편안', '따뜻', '따듯',
    '무난', '어울리', '적당', '넉넉',
    '깔끔', '심플', '간단', '간편',
    '코디', '매치', '데일리',
}
def categorize_keyword(word):
    """키워드를 핵심/보조/기타로 분류"""
    parts = word.split(' ')
    if len(parts) == 1:
        if word in CORE_ATTRIBUTES: return 'core'
        if word in SECONDARY_TERMS: return 'secondary'
        return 'other'
    # bigram: 핵심어가 하나라도 있으면 core
    if any(p in CORE_ATTRIBUTES for p in parts): return 'core'
    if all(p in SECONDARY_TERMS for p in parts): return 'secondary'
    return 'other'
print(f'핵심 속성어: {len(CORE_ATTRIBUTES)}개')
print(f'보조 표현:   {len(SECONDARY_TERMS)}개')

핵심 속성어: 192개
보조 표현:   15개


In [25]:
# 전체 토픽 키워드 출력 (핵심/보조/기타 분류)
print(f'전체 토픽 수: {len(topic_keywords)}개')
print('=' * 100)
for tid in sorted(topic_keywords.keys()):
    size = int(cluster_sizes.get(tid, 0))
    core, secondary, other = [], [], []
    for word, score in topic_keywords[tid]:
        cat = categorize_keyword(word)
        if cat == 'core': core.append(word)
        elif cat == 'secondary': secondary.append(word)
        else: other.append(word)
    print(f'\n[T{tid:>3}] ({size:>6,}건)')
    if core:      print(f'  🟢 핵심: {", ".join(core[:8])}')
    if secondary: print(f'  🟡 보조: {", ".join(secondary[:6])}')
    if other:     print(f'  ⚪ 기타: {", ".join(other[:6])}')

전체 토픽 수: 68개

[T  0] ( 2,379건)
  🟢 핵심: 동생 사이즈, 동생 핏
  ⚪ 기타: 동생, 남동생, 동생 선물, 동생 좋아하, 동생 생일, 남동생 선물

[T  1] ( 4,980건)
  🟢 핵심: 아들 사이즈
  ⚪ 기타: 아들, 아이, 중학, 아들 좋아하, 딸아이, 중학 아들

[T  2] (28,411건)
  🟢 핵심: 배송 빠르, 빠르, 배송, 느리, 배송 느리, 걸리, 배송 걸리, 빠르 사이즈
  ⚪ 기타: 기다리, 예약, 포장

[T  3] ( 1,649건)
  🟢 핵심: 선물 사이즈, 선물 색감, 선물 핏
  ⚪ 기타: 선물, 엄마, 어머니, 선물 좋아하, 선물 사람, 지인 선물

[T  4] ( 3,622건)
  🟢 핵심: 맨투맨, 기본 맨투맨, 오버핏 맨투맨, 무난 맨투맨, 맨투맨 핏, 가성비 맨투맨, 맨투맨 필요, 맨투맨 레이어드

[T  5] ( 2,471건)
  🟢 핵심: 반바지, 여름 반바지, 편하 반바지, 반바지 필요, 가성비 반바지, 기본 반바지, 무난 반바지, 반바지 가격

[T  6] (30,248건)
  🟢 핵심: 바지, 팬츠, 바지 핏, 청바지, 바지 편하, 카고, 데님, 바지 어울리
  ⚪ 기타: 카고팬츠, 벌룬팬츠, 와이드, 카고바지

[T  7] ( 2,277건)
  🟢 핵심: 핏 운동, 운동 사이즈, 가성비 운동, 재질 운동
  ⚪ 기타: 운동, 운동 편하, 운동복, 헬스, 운동 평소, 헬스장

[T  8] ( 2,301건)
  🟢 핵심: 겨울 교복, 핏 교복
  ⚪ 기타: 교복, 학교, 등산, 교복템, 교복 어울리, 학원

[T  9] ( 2,130건)
  🟢 핵심: 핏 친구, 선물 사이즈, 사이즈 친구, 색감 친구
  ⚪ 기타: 친구, 생일, 생일 선물, 친구 생일, 친구 선물, 선물

[T 10] ( 3,124건)
  🟢 핵심: 친구 사이즈, 선물 사이즈
  ⚪ 기타: 남자 친구, 남자, 남친, 친구, 친구 선물, 친구 커플

[T 11] ( 2,540건)
  🟢 핵심: 커플 후드티, 커플 

## 6-10. 범용 속성어 후처리 (v3) — 시각화/보고서 전용

> ! **이 처리는 군집 결과를 바꾸지 않습니다.**  
> 원본 c-TF-IDF 점수와 키워드(`topic_keywords`)는 그대로 보존.  
> **보고서·시각화에서 보여줄 대표어만** 범용 속성어를 제한합니다.

**v3 전략:** 6-8 기본 필터링 결과에서 범용 속성어(`GENERIC_ATTRS`)가 포함된 키워드를  
토픽당 최대 `MAX_GENERAL=2`개로 제한. 빈 슬롯은 다음 순위 핵심 키워드로 채움.

**예외 규칙:** 1위 키워드 자체가 범용 속성어인 토픽 → 해당 토픽은 속성 자체가 핵심이므로 `MAX_GENERAL=3`으로 완화.

**최종 라벨 결정:** 후처리 대표어 + 대표 리뷰 원문을 함께 보고 **수동 지정**.

| 단계 | 변수 | 설명 |
|---|---|---|
| 6-6 | `topic_keywords_raw` | c-TF-IDF 원점수 TOP 30 |
| 6-8 | `topic_keywords` | 반복/범용쌍/어순중복 제거 (보존) |
| **6-4-I** | **`topic_keywords_v3`** | **범용 속성어 최대 2개 제한 (표시용)** |


In [26]:
# ─────────────────────────────────────────────────────────────
# display 키워드 생성 — 시각화/보고서 전용 후처리 (군집 결과 변경 아님)
# 원본: topic_keywords (6-8 기본 필터링 결과, 보존)
# 추가 후처리: 범용 속성어 토픽당 MAX_GENERAL개 제한
# ─────────────────────────────────────────────────────────────

# GENERIC_ATTRS(9개)와 별도 — 정규화 후 vocab에 남는 대표 형태 4개만 타겟
DISPLAY_FILTER_ATTRS = {'편하', '따뜻', '무난', '어울리'}

MAX_GENERAL      = 2   # 기본: 토픽당 범용 속성어 최대 허용 수
MAX_GENERAL_CORE = 3   # 예외: 1위 키워드가 범용 속성어인 토픽 → 1개 추가 허용
TOP_N_V3         = TOP_N_FINAL

topic_keywords_v3 = {}
exception_topics  = []

for topic_id in topic_id_order:
    raw = topic_keywords[topic_id]
    top_word      = raw[0][0] if raw else ''
    is_core_topic = bool(set(top_word.split()) & DISPLAY_FILTER_ATTRS)
    max_g         = MAX_GENERAL_CORE if is_core_topic else MAX_GENERAL
    if is_core_topic:
        exception_topics.append((topic_id, top_word))
    filtered      = []
    general_count = 0
    for word, score in raw:
        if len(filtered) >= TOP_N_V3:
            break
        is_general = bool(set(word.split()) & DISPLAY_FILTER_ATTRS)
        if is_general:
            if general_count < max_g:
                filtered.append((word, score))
                general_count += 1
        else:
            filtered.append((word, score))
    topic_keywords_v3[topic_id] = filtered

print(f'display 키워드 생성 완료: {len(topic_keywords_v3)}개 토픽')
print(f'DISPLAY_FILTER_ATTRS: {DISPLAY_FILTER_ATTRS}')
print(f'기본: MAX_GENERAL={MAX_GENERAL} / 예외(속성 핵심 토픽): MAX_GENERAL={MAX_GENERAL_CORE}')
print(f'\n예외 적용 토픽 ({len(exception_topics)}개) — 1위 키워드가 범용 속성어')
print('=' * 80)
for etid, eword in exception_topics:
    size = int(cluster_sizes.get(etid, 0))
    core, secondary, other = [], [], []
    for word, _ in topic_keywords_v3[etid]:
        cat = categorize_keyword(word)
        if cat == 'core':        core.append(word)
        elif cat == 'secondary': secondary.append(word)
        else:                    other.append(word)
    print(f'\n[T{etid:>3}] ({size:,}건)  top="{eword}" ★예외')
    if core:      print(f'  🟢 핵심: {", ".join(core[:8])}')
    if secondary: print(f'  🟡 보조: {", ".join(secondary[:6])}')
    if other:     print(f'  ⚪ 기타: {", ".join(other[:6])}')


display 키워드 생성 완료: 68개 토픽
DISPLAY_FILTER_ATTRS: {'편하', '따뜻', '무난', '어울리'}
기본: MAX_GENERAL=2 / 예외(속성 핵심 토픽): MAX_GENERAL=3

예외 적용 토픽 (5개) — 1위 키워드가 범용 속성어

[T 12] (12,355건)  top="따뜻" ★예외
  🟢 핵심: 따뜻 핏, 사이즈 따뜻, 보온, 포근
  🟡 보조: 따뜻

[T 13] (18,682건)  top="기모 따뜻" ★예외
  🟢 핵심: 기모 따뜻, 기모, 기모 겨울, 핏 기모, 기모 들어가, 오버핏 기모, 사이즈 기모, 편하 기모
  ⚪ 기타: 안감, 들어가 따뜻

[T 53] (6,481건)  top="가성비 편하" ★예외
  🟢 핵심: 가성비 편하, 편하 가격, 가격 저렴하, 저렴, 가격 대비, 무난 가격, 가격 저렴, 대비
  ⚪ 기타: 저렴하, 착하

[T 60] (8,279건)  top="오버핏 편하" ★예외
  🟢 핵심: 오버핏 편하, 핏 오버핏, 오버핏 색감, 사이즈 오버핏, 적당 오버핏, 오버핏, 오버핏 무난, 오버핏 떨어지

[T 65] (2,635건)  top="두께감 편하" ★예외
  🟢 핵심: 두께감 편하, 두께 적당, 두께, 두께 편하, 이너, 두께감, 두께감 적당, 두툼 편하
  🟡 보조: 적당
  ⚪ 기타: 비치, 걸치, 여기저기


## 6-11. 원본 vs v3 비교 출력

범용 속성어 제한으로 **변화가 생긴 토픽만** 출력해 효과 확인.


In [27]:
print('원본(6-8) vs display(범용속성어 제한) — 변화 토픽만 출력')
print('=' * 100)

changed = 0
for tid in sorted(topic_keywords.keys()):
    orig_kws = [w for w, _ in topic_keywords[tid]]
    v3_kws   = [w for w, _ in topic_keywords_v3[tid]]
    if orig_kws != v3_kws:
        size = int(cluster_sizes.get(tid, 0))
        exc  = ' ★예외' if any(tid == e[0] for e in exception_topics) else ''
        print(f'\n[T{tid:>3}] ({size:,}건){exc}')
        print(f'  원본: {" | ".join(orig_kws[:10])}')
        print(f'  display: {" | ".join(v3_kws[:10])}')
        core, secondary, other = [], [], []
        for word, _ in topic_keywords_v3[tid]:
            cat = categorize_keyword(word)
            if cat == 'core':        core.append(word)
            elif cat == 'secondary': secondary.append(word)
            else:                    other.append(word)
        if core:      print(f'  🟢 핵심: {", ".join(core[:8])}')
        if secondary: print(f'  🟡 보조: {", ".join(secondary[:6])}')
        if other:     print(f'  ⚪ 기타: {", ".join(other[:6])}')
        changed += 1

print(f'\n변화 있는 토픽: {changed}개 / {len(topic_keywords)}개')


원본(6-8) vs display(범용속성어 제한) — 변화 토픽만 출력

[T  3] (1,649건)
  원본: 선물 | 엄마 | 어머니 | 선물 좋아하 | 선물 사이즈 | 선물 사람 | 지인 선물 | 지인 | 부모 | 선물 따뜻
  display: 선물 | 엄마 | 어머니 | 선물 좋아하 | 선물 사이즈 | 선물 사람 | 지인 선물 | 지인 | 부모 | 선물 따뜻
  🟢 핵심: 선물 사이즈, 선물 색감, 선물 핏
  ⚪ 기타: 선물, 엄마, 어머니, 선물 좋아하, 선물 사람, 지인 선물

[T  5] (2,471건)
  원본: 반바지 | 여름 반바지 | 편하 반바지 | 반바지 필요 | 가성비 반바지 | 기본 반바지 | 무난 반바지 | 반바지 가격 | 반바지 어울리 | 반바지 시원
  display: 반바지 | 여름 반바지 | 편하 반바지 | 반바지 필요 | 가성비 반바지 | 기본 반바지 | 무난 반바지 | 반바지 가격 | 반바지 시원 | 반바지 핏
  🟢 핵심: 반바지, 여름 반바지, 편하 반바지, 반바지 필요, 가성비 반바지, 기본 반바지, 무난 반바지, 반바지 가격

[T  8] (2,301건)
  원본: 교복 | 학교 | 등산 | 교복템 | 교복 어울리 | 학원 | 학생 | 학교 편하 | 공부 | 교복 편하
  display: 교복 | 학교 | 등산 | 교복템 | 교복 어울리 | 학원 | 학생 | 학교 편하 | 공부 | 독서실
  🟢 핵심: 겨울 교복, 핏 교복
  ⚪ 기타: 교복, 학교, 등산, 교복템, 교복 어울리, 학원

[T 12] (12,355건) ★예외
  원본: 따뜻 | 따뜻 핏 | 사이즈 따뜻 | 보온 | 두툼 따뜻 | 재질 따뜻 | 오버핏 따뜻 | 따뜻 가격 | 두께감 따뜻 | 따뜻 가성비
  display: 따뜻 | 따뜻 핏 | 사이즈 따뜻 | 보온 | 포근
  🟢 핵심: 따뜻 핏, 사이즈 따뜻, 보온, 포근
  🟡 보조: 따뜻

[T 16] (1,519건)
  원본: 동네 | 마실 | 잠옷 | 나가 | 동네 마실 | 편의점 | 나가

## 저장

### 저장 파일 목록

| 파일명 | 내용 | 비고 |
|---|---|---|
| `step6_1_topic_keywords.csv` | 기본 필터링 키워드 (카테고리 포함) | **원본 보존** |
| `step6_2_topic_keywords_display.csv` | 범용 속성어 제한 키워드 | 보고서·시각화 표시용 |
| `step6_3_exception_topics_display.csv` | 예외 적용 토픽 목록 | 수동 라벨링 참고 |
| `step6_4_topic_id_order.npy` | 토픽 순서 배열 | |
| `step6_5_count_matrix.npz` | Count 행렬 | |
| `step6_6_words.npy` | 단어 목록 | |
| `step6_7_vectorizer.pkl` | CountVectorizer | |
| `step6_8_ctfidf.pkl` | c-TF-IDF 행렬 + 단어 + 토픽 매핑 | |
| `step6_9_stopwords.pkl` | 불용어 셋 | |
| `step6_10_normalize_map.pkl` | 정규화 맵 | |
| `step6_11_core_attributes.pkl` | 핵심 속성어 | |
| `step6_12_secondary_terms.pkl` | 보조 표현 | |
| `step6_13_generic_attrs.pkl` | 범용 속성어 | |

> **원본 보존 원칙:** `step6_1_topic_keywords.csv`는 군집 분석 재현용으로 보존.  
> `step6_2_topic_keywords_display.csv`는 사람이 보는 대표어 후처리 결과로만 사용.

In [28]:
# ── 원본 토픽 키워드 CSV (카테고리 포함)
rows = []
for topic_id in sorted(topic_keywords.keys()):
    size = int(cluster_sizes.get(topic_id, 0))
    for rank, (word, score) in enumerate(topic_keywords[topic_id], 1):
        rows.append({
            'topic_id':     topic_id,
            'topic_size':   size,
            'rank':         rank,
            'word':         word,
            'category':     categorize_keyword(word),
            'ctfidf_score': round(float(score), 4),
        })
pd.DataFrame(rows).to_csv(
    OUTPUT_DIR + 'step6_1_topic_keywords.csv', index=False, encoding='utf-8-sig'
)

# ── v3 토픽 키워드 CSV
rows_v3 = []
for topic_id in sorted(topic_keywords_v3.keys()):
    size = int(cluster_sizes.get(topic_id, 0))
    for rank, (word, score) in enumerate(topic_keywords_v3[topic_id], 1):
        rows_v3.append({
            'topic_id':     topic_id,
            'topic_size':   size,
            'rank':         rank,
            'word':         word,
            'category':     categorize_keyword(word),
            'ctfidf_score': round(float(score), 4),
        })
pd.DataFrame(rows_v3).to_csv(
    OUTPUT_DIR + 'step6_2_topic_keywords_display.csv', index=False, encoding='utf-8-sig'
)

# ── 예외 토픽 CSV (1위 키워드가 범용 속성어 → MAX_GENERAL_CORE 적용)
df_exc = pd.DataFrame(exception_topics, columns=['topic_id', 'top_keyword'])
df_exc['topic_size'] = df_exc['topic_id'].map(lambda t: int(cluster_sizes.get(t, 0)))
df_exc['v3_keywords'] = df_exc['topic_id'].map(
    lambda t: ' | '.join(w for w, _ in topic_keywords_v3.get(t, []))
)
df_exc.to_csv(OUTPUT_DIR + 'step6_3_exception_topics_display.csv', index=False, encoding='utf-8-sig')

# ── 나머지 파일
import numpy as np
from scipy.sparse import save_npz
np.save(OUTPUT_DIR + 'step6_4_topic_id_order.npy', np.array(topic_id_order))
save_npz(OUTPUT_DIR + 'step6_5_count_matrix.npz', count_matrix)
np.save(OUTPUT_DIR + 'step6_6_words.npy', words)
with open(OUTPUT_DIR + 'step6_7_vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)
with open(OUTPUT_DIR + 'step6_8_ctfidf.pkl', 'wb') as f:
    pickle.dump((ctfidf_matrix, words, topic_id_order), f)
with open(OUTPUT_DIR + 'step6_9_stopwords.pkl', 'wb') as f:
    pickle.dump(STOPWORDS, f)
with open(OUTPUT_DIR + 'step6_10_normalize_map.pkl', 'wb') as f:
    pickle.dump(NORMALIZE_MAP, f)
with open(OUTPUT_DIR + 'step6_11_core_attributes.pkl', 'wb') as f:
    pickle.dump(CORE_ATTRIBUTES, f)
with open(OUTPUT_DIR + 'step6_12_secondary_terms.pkl', 'wb') as f:
    pickle.dump(SECONDARY_TERMS, f)
with open(OUTPUT_DIR + 'step6_13_generic_attrs.pkl', 'wb') as f:
    pickle.dump(GENERIC_ATTRS, f)

print('저장 완료:')
print(f'  step6_1_topic_keywords.csv        ({len(rows)}행, 원본 키워드)')
print(f'  step6_2_topic_keywords_display.csv     ({len(rows_v3)}행, 후처리 키워드)')
print(f'  step6_3_exception_topics_display.csv   ({len(exception_topics)}개 예외 토픽)')
print('  step6_4 ~ step6_13 pkl/npy 저장 완료')


저장 완료:
  step6_1_topic_keywords.csv        (1020행, 원본 키워드)
  step6_2_topic_keywords_display.csv     (983행, 후처리 키워드)
  step6_3_exception_topics_display.csv   (5개 예외 토픽)
  step6_4 ~ step6_13 pkl/npy 저장 완료


---
# 확인용

In [29]:
# 보호어 vocabulary 생존 여부 점검
vocab = set(vectorizer.get_feature_names_out())

survived = [w for w in all_protected if w in vocab]
dropped = [w for w in all_protected if w not in vocab]

print(f'보호어 총 {len(all_protected)}개')
print(f'  vocabulary 생존: {len(survived)}개')
print(f'  vocabulary 탈락: {len(dropped)}개')

if dropped:
    print(f'\n[탈락 보호어 목록] → min_df={FINAL_MIN_DF} 기준 미달')
    for w in dropped:
        print(f'  {w}')

보호어 총 104개
  vocabulary 생존: 98개
  vocabulary 탈락: 6개

[탈락 보호어 목록] → min_df=20 기준 미달
  박시핏
  톤업
  쨍함
  비쌈
  창렬
  끼임
